# COLLECT THAI PREDICTION DATASET
Run in Google Colab and export data into Google Drive

## GEE Setup

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project=PROJECT_NAME)

## Select HLSS and Filter
Tile 47QMV at 2024

In [ ]:
def filter(tile_list, i_date, f_date, cloud_max, spatial_min):
    hlss30 = ee.ImageCollection("NASA/HLS/HLSS30/v002")
    hlss30_filter = hlss30.filter(ee.Filter.inList("MGRS_TILE_ID", tile_list))
    hlss30_filter = hlss30_filter.filterDate(i_date, f_date)
    hlss30_filter = hlss30_filter.filter(ee.Filter.lte("CLOUD_COVERAGE", cloud_max))
    hlss30_filter = hlss30_filter.filter(ee.Filter.gte("SPATIAL_COVERAGE", spatial_min))
    hlss30_filter = hlss30_filter.select(["B2", "B3", "B4", "B8A", "B11", "B12"])

    return hlss30_filter

In [ ]:
tile_list = ["47QMV"]
prediction = filter(tile_list = tile_list,
                    i_date = "2024-01",
                    f_date = "2025-01",
                    cloud_max = 20,
                    spatial_min = 50)
prediction.size().getInfo()

In [ ]:
prediction_index = prediction.aggregate_array("system:index").getInfo()
prediction_index

## Visualization

In [ ]:
def normalize_band(image):
    min_list = [0.01, 0.01, 0.01, 0.01, 0.005, 0.005]
    max_list = [0.18, 0.18, 0.18, 0.5, 0.35, 0.3]

    band_names = ["B2", "B3", "B4", "B8A", "B11", "B12"]
    normalized_bands = []
    for band, min_val, max_val in zip(band_names, min_list, max_list):
        band_img = image.select(band)
        norm_band = band_img.subtract(min_val).divide(max_val - min_val).clamp(0,1).rename(band)
        normalized_bands.append(norm_band)

    normalized_image = ee.Image(normalized_bands)
    normalized_image = normalized_image.unmask(0)

    return normalized_image

In [ ]:
def create_layer(image, title, bands):
  vis_params = {
      "min": 0,
      "max": 1,
      "bands": bands
  }
  layer = geemap.ee_tile_layer(image, vis_params, name=title)

  return layer

In [ ]:
hlss30 = ee.ImageCollection("NASA/HLS/HLSS30/v002")
Map1 = geemap.Map(center=(17.7, 98.3), zoom=9)
for index in prediction_index[:11]:
    image = hlss30.filter(ee.Filter.eq("system:index", index))
    image = image.select(["B2", "B3", "B4", "B8A", "B11", "B12"])
    normalized_image = normalize_band(image.first())
    layer = create_layer(normalized_image, index.split("_")[1], bands=["B4", "B3", "B2"])
    Map1.add_layer(layer)

Map1

In [ ]:
hlss30 = ee.ImageCollection("NASA/HLS/HLSS30/v002")
Map2 = geemap.Map(center=(17.7, 98.3), zoom=9)
for index in prediction_index[11:]:
    image = hlss30.filter(ee.Filter.eq("system:index", index))
    image = image.select(["B2", "B3", "B4", "B8A", "B11", "B12"])
    normalized_image = normalize_band(image.first())
    layer = create_layer(normalized_image, index.split("_")[1], bands=["B4", "B3", "B2"])
    Map2.add_layer(layer)

Map2

## Export data

In [ ]:
def export(image_collection, folder):
  def export_image(image, folder):
      no_data_val = 0.0
      image = image.unmask(no_data_val)
      export_task = ee.batch.Export.image.toDrive(
          image = image.toFloat(),
          description = f"{image.get("id").getInfo()}",
          folder = folder,
          fileNamePrefix = image.get("id").getInfo(),
          fileFormat = "GeoTIFF",
          region = image.geometry(),
          scale=30,
          formatOptions={"noData": no_data_val},
      )
      export_task.start()

  def normalize_band(image):
    min_list = [0.01, 0.01, 0.01, 0.01, 0.005, 0.005]
    max_list = [0.18, 0.18, 0.18, 0.5, 0.35, 0.3]

    # Normalize
    band_names = ["B2", "B3", "B4", "B8A", "B11", "B12"]
    normalized_bands = []
    for band, min_val, max_val in zip(band_names, min_list, max_list):
        band_img = image.select(band)
        norm_band = band_img.subtract(min_val).divide(max_val - min_val)
        normalized_bands.append(norm_band)

    # Combine to img
    normalized_image = ee.Image.cat(normalized_bands).rename(band_names)
    normalized_image = normalized_image.clamp(0,1)
    normalized_image = normalized_image.unmask(0.0)
    normalized_image = normalized_image.set("id", image.id())

    return normalized_image

  # Exports
  bands = ["B2", "B3", "B4", "B8A", "B11", "B12"]
  for index, image in enumerate(image_collection.toList(image_collection.size()).getInfo()):
    image_ee = ee.Image(image["id"]).select(bands)
    image_ee_normalized = normalize_band(image_ee)
    export_image(image_ee_normalized, folder)

In [ ]:
hlss30 = ee.ImageCollection("NASA/HLS/HLSS30/v002")
hlss30 = hlss30.select(["B2", "B3", "B4", "B8A", "B11", "B12"])
hlss30_predict = hlss30.filter(ee.Filter.inList("system:index", prediction_index))
print(f"Prediction set size: {hlss30_predict.size().getInfo()}")

In [ ]:
export(image_collection=hlss30_predict, folder=FOLDER_NAME)